## Initialize Objective ##

In [ ]:
import numpy as np
from modopt.core.moo_problem import MOO_Problem


class Rosenbrock2D(MOO_Problem):
    def initialize(self, ):
        # Name your problem
        self.problem_name = 'Rosenbrock2D'

    def setup(self):
        # Add design variables of your problem
        self.add_design_variables('x',
                                  shape=(2, ),
                                  vals=np.array([.3, .3]))
        self.add_objectives(['f1', 'f2'])


    def setup_derivatives(self):
        # Declare objective gradient and its shape
        self.declare_objectives_gradient(wrt='x')

    # Compute the value of the objective with given design variable values
    def compute_objectives(self, dvs, objs):
        x, y = dvs['x']

        objs['f1'] = (1 - x)**2 + 100 * (y - x**2)**2  # Rosenbrock function
        objs['f2'] = x**2 + y**2  # Sphere function (2nd objective)

    def compute_objectives_gradient(self, dvs, grads):
        x, y = dvs['x']
        
        grads['f1'] = np.array([
            -400 * x * (y - x**2) + 2 * (x - 1),
            200 * (y - x**2)
        ])

        grads['f2'] = np.array([2*x, 2*y])

## Initialize Optimizer ##

In [4]:
import time
from modopt import Optimizer
from deap import base, creator, tools
import random


class NSGAII(Optimizer):


    def initialize(self):

        # Name your algorithm
        self.solver_name = 'NSGA-II'

        self.obj = self.problem._compute_objectives
        self.grad = self.problem._compute_objectives_gradient  # If using gradients


        self.options.declare('maxiter', default=1000, types=int)
        self.options.declare('opt_tol', default=1e-5, types=float)
        self.options.declare('initialPopulationSize', default=200, types=int)
        self.options.declare('rangeLow', default = -4.0, types = float)
        self.options.declare('rangeHigh', default =  4.0, types = float)
        self.options.declare('mutationRateGene', default = 0.1, types = float)
        self.options.declare('alpha', default = 0.5, types = float)
        self.options.declare('tournsize', default = 3, types = int)
        self.options.declare('cxProb' , default =  0.5, types = float)
        self.options.declare('mutationRateInd', default =   0.2, types = float)

        # Enable user to specify, as a list, which among the available outputs
        # need to be written to output files
        self.options.declare('readable_outputs', types=list, default=[])

        # Specify format of outputs available from your optimizer after each iteration
        self.available_outputs = {
            'itr': int,
            'obj': float,
            # for arrays from each iteration, shapes need to be declared
            'x': (float, (self.problem.nx, )),
            'opt': float,
            'time': float,
        }

    def setup(self):
        if not hasattr(creator, "FitnessMin"):
            # -1.0 Weight means minimization, 1.0 for Maximization
            creator.create("FitnessMin", base.Fitness, weights=(-1.0, -1.0))
        if not hasattr(creator, "Individual"):
            creator.create("Individual", list, fitness=creator.FitnessMin)

        IND_SIZE = self.problem.nx

        self.toolbox = base.Toolbox()
        self.toolbox.register("attribute", random.uniform, self.options['rangeLow'], self.options['rangeHigh'])  # Adjusted range
        self.toolbox.register("individual", tools.initRepeat, creator.Individual,
                 self.toolbox.attribute, n=IND_SIZE)
        self.toolbox.register("population", tools.initRepeat, list, self.toolbox.individual)

        self.toolbox.register("mate", tools.cxBlend, alpha=self.options['alpha'])
        self.toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=(self.options['rangeHigh'] - self.options['rangeLow'])*0.1, indpb=self.options['mutationRateGene'])
        self.toolbox.register("select", tools.selNSGA2)
        def modopt_evaluate(individual):
            return tuple(self.problem._compute_objectives(individual))  # Ensure it returns multiple objectives

        self.toolbox.register("evaluate", modopt_evaluate)

    def solve(self):
        x = self.problem.x0
        opt_tol = self.options['opt_tol']
        maxiter = self.options['maxiter']

        obj = self.obj
        grad = self.grad

        start_time = time.time()

        # Setting intial values for initial iterates
        x_k = x * 1.
        f_k = obj(x_k)
        g_k = grad(x_k)

        # Iteration counter
        itr = 0

        # Optimality
        opt = float('inf')

        # Initializing outputs
        self.update_outputs(itr=0,
                            x=x_k,
                            obj=f_k,
                            opt=opt,
                            time=time.time() - start_time)

        pop = self.toolbox.population(n=self.options['initialPopulationSize'])
        CXPB, MUTPB, NGEN = self.options['cxProb'], self.options['mutationRateInd'], maxiter

        # Evaluate the entire population
        for ind in pop:
            ind.fitness.values = self.toolbox.evaluate(ind)

        while (opt > opt_tol and itr < NGEN):
            offspring = list(map(self.toolbox.clone, pop))

            # Apply crossover and mutation on the offspring
            for child1, child2 in zip(offspring[::2], offspring[1::2]):
                if random.random() < CXPB:
                    self.toolbox.mate(child1, child2)
                    del child1.fitness.values
                    del child2.fitness.values

            for mutant in offspring:
                if random.random() < MUTPB:
                    self.toolbox.mutate(mutant)
                    del mutant.fitness.values

            # Evaluate the individuals with an invalid fitness
            invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
            for ind in invalid_ind:
                ind.fitness.values = self.toolbox.evaluate(ind)

            # The population is entirely replaced by the offspring
            combined_pop = pop + offspring

            pop[:] = self.toolbox.select(combined_pop, len(pop))

            # Output the best solution(s)
            best_inds = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]

            # Convert Pareto front to NumPy arrays
            x_k = [np.array(ind) for ind in best_inds]
            f_k = [ind.fitness.values for ind in best_inds]

            itr += 1

            # Append arrays inside outputs dict with new values from the current iteration
            self.update_outputs(itr=itr,
                                x=x_k,
                                obj=f_k,
                                opt=f_k if isinstance(f_k, float) else min(f_k),  # Track best objective if needed
                                time=time.time() - start_time)


        self.total_time = time.time() - start_time

        self.results = {
            'x': x_k,
            'objective': f_k,
            'optimality': f_k if isinstance(f_k, float) else min(f_k),
            'itr': itr,
            'time': self.total_time
        }

        # Run post-processing for the Optimizer() base class
        self.run_post_processing()

        return self.results

## Test ##

In [5]:
# Set your optimality tolerance
opt_tol = 1E-8
# Set maximum optimizer iteration limit
maxiter = 1000

prob = Rosenbrock2D()

# Set up your optimizer with your problem and pass in optimizer parameters
# And declare outputs to be stored
optimizer = NSGAII(prob,
                        opt_tol=opt_tol,
                        maxiter=maxiter,
                        readable_outputs=['itr', 'obj', 'x', 'opt', 'time'])

# Solve your optimization problem
optimizer.solve()

# Print results of optimization (summary_table contains information from each iteration)
optimizer.print_results(summary_table=True)

# Print any output that was declared
# Since the arrays are long, here we only print the last entry and
# verify it with the print_results() above

print("\nPareto-optimal solutions (design variables):")
for x_sol in optimizer.results['x']:
    print(x_sol)

print("\nPareto-optimal objectives:")
for f_sol in optimizer.results['objective']:
    print(f_sol)

print("\nFinal iteration count:", optimizer.results['itr'])
print("Total optimization time:", optimizer.results['time'])

Setting objective names as f1, f2.


AttributeError: 'Rosenbrock2D' object has no attribute 'declare_objectives_gradient'